# Session 4 — Paragraph-Level Analysis
## Measure 1: Paragraph Semantic Coherence

In this notebook, you will:
- split your text into paragraphs and sentences
- compute **embedding-based coherence** per paragraph using MiniLM
- compare coherence distributions across two books
- connect this to how RAG systems and LLMs assess chunk quality

We use Stephen King's *Pet Sematary* and *The Shining* as examples.



In [61]:
import re
from pathlib import Path

from pathlib import Path

files = list(Path("/mnt/data").glob("*.txt"))
for file in files:
    print(file.name)


In [62]:
import re
from pathlib import Path

def basic_normalize(text: str) -> str:
    """
    Basic normalization:
    - normalize line endings
    - strip trailing spaces
    - collapse 3+ blank lines into 2
    """
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [ln.rstrip() for ln in text.split('\n')]
    text = "\n".join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text

def remove_decorative_lines(text: str) -> str:
    """
    Remove lines that are mostly decoration (* * *, -----, boxes, etc.).
    """
    cleaned_lines = []
    for ln in text.split('\n'):
        stripped = ln.strip()

        # keep empty lines (paragraph breaks)
        if stripped == "":
            cleaned_lines.append("")
            continue

        letters = sum(ch.isalpha() for ch in stripped)
        non_alnum = sum(not ch.isalnum() for ch in stripped)

        # very short & mostly non-alphanumeric → decoration
        if len(stripped) <= 3 and non_alnum > 0:
            continue

        # long but with no letters and many symbols → decoration
        if letters == 0 and non_alnum > 3:
            continue

        cleaned_lines.append(ln)

    return "\n".join(cleaned_lines)

def clean_pet_sematary(raw: str) -> str:
    """
    Clean Pet Sematary:
    - cut everything before the REAL 'Chapter 1' heading
    - normalize + remove decorative lines
    """
    m = re.search(r'^\s*Chapter\s+1\s*$', raw, re.MULTILINE)
    if m:
        raw = raw[m.start():]
    else:
        print("Warning: standalone 'Chapter 1' not found; using full text.")

    text = basic_normalize(raw)
    text = remove_decorative_lines(text)
    return text

def clean_the_shining(raw: str) -> str:
    """
    Clean The Shining:
    - cut everything before the first '<< 1 >>' marker
    - normalize + remove decorative lines
    """
    m = re.search(r'<<\s*1\s*>>', raw)
    if m:
        raw = raw[m.start():]
    else:
        print("Warning: '<< 1 >>' not found; using full text.")

    text = basic_normalize(raw)
    text = remove_decorative_lines(text)
    return text


In [63]:
from pathlib import Path

# Paths – adapt if your files are elsewhere on YOUR machine
pet_path     = Path("../data/PetSemetary.txt")
shining_path = Path("../data/TheShining.txt")

# Load raw files
pet_raw     = pet_path.read_text(encoding="utf-8", errors="replace")
shining_raw = shining_path.read_text(encoding="utf-8", errors="replace")

# Clean
pet_clean     = clean_pet_sematary(pet_raw)
shining_clean = clean_the_shining(shining_raw)

print("Pet Sematary cleaned length:", len(pet_clean))
print("The Shining cleaned length:", len(shining_clean))

# Save cleaned versions (recommended)
pet_clean_path     = Path("../data/PetSemetary_clean.txt")
shining_clean_path = Path("../data/TheShining_clean.txt")

pet_clean_path.write_text(pet_clean, encoding="utf-8")
shining_clean_path.write_text(shining_clean, encoding="utf-8")

# Quick sanity check: show first ~20 lines
print("=== Pet Sematary (cleaned) sample ===")
print("\n".join(pet_clean.splitlines()[:20]))

print("\n=== The Shining (cleaned) sample ===")
print("\n".join(shining_clean.splitlines()[:20]))


Pet Sematary cleaned length: 803956
The Shining cleaned length: 898577
=== Pet Sematary (cleaned) sample ===


                                 Chapter 1

           Louis Creed, who had lost his father at three and who had never known
a grandfather, never expected to find a father as he entered his middle age, but
that was exactly what happened… although he called this man a friend, as a
grown man must do when he finds the man who should have been his father
relatively late in life. He met this man on the evening he and his wife and his two
children moved into the big white frame house in Ludlow. Winston Churchill
moved in with them. Church was his daughter Eileen’s cat.
   The search committee at the University had moved slowly, the search for a
house within commuting distance of the University had been hair-raising, and by
the time they neared the place where he believed the house to be (all the
landmarks are right… like the astrological signs the night before Caesar was
assassinate

In [64]:
from pathlib import Path

pet_text     = Path("../data/PetSemetary_clean.txt").read_text(encoding="utf-8")
shining_text = Path("../data/TheShining_clean.txt").read_text(encoding="utf-8")

# now use pet_text, shining_text
# e.g.
# pet_paras = split_into_paragraphs(pet_text)
# shining_paras = split_into_paragraphs(shining_text)


In [65]:
import re
from typing import List, Tuple
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Paragraph split
def split_into_paragraphs(text: str, min_words: int = 10) -> List[str]:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    raw_paras = re.split(r'\n\s*\n+', text)
    paras = []
    for p in raw_paras:
        cleaned = re.sub(r'\s+', ' ', p).strip()
        if not cleaned:
            continue
        if len(cleaned.split()) < min_words:
            continue
        paras.append(cleaned)
    return paras

# 2. Sentence split
def sentence_split(paragraph: str) -> List[str]:
    sentences = re.split(r'[.!?]+\s+', paragraph.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

# 3. Cosine sim + coherence
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    if a.ndim > 1:
        a = a.reshape(-1)
    if b.ndim > 1:
        b = b.reshape(-1)
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

model = SentenceTransformer("all-MiniLM-L6-v2")

def paragraph_coherence_embeddings(paragraphs: List[str]) -> Tuple[list, list]:
    scores = []
    lengths = []
    for p in paragraphs:
        sents = sentence_split(p)
        if len(sents) < 2:
            continue
        sent_embs = model.encode(sents)
        centroid = sent_embs.mean(axis=0)
        sims = [cosine_similarity(e, centroid) for e in sent_embs]
        scores.append(sum(sims) / len(sims))
        lengths.append(len(" ".join(sents).split()))
    return scores, lengths

# Build paragraphs and coherence scores
pet_paras     = split_into_paragraphs(pet_text)
shining_paras = split_into_paragraphs(shining_text)

pet_scores, pet_lengths         = paragraph_coherence_embeddings(pet_paras)
shining_scores, shining_lengths = paragraph_coherence_embeddings(shining_paras)


In [66]:
import numpy as np

print("="*60)
print("PARAGRAPH COHERENCE SUMMARY")
print("="*60)

print(f"Pet Sematary paragraphs used:     {len(pet_scores)}")
print(f"The Shining paragraphs used:      {len(shining_scores)}")
print()

print(f"Pet Sematary mean coherence:      {np.mean(pet_scores):.3f}")
print(f"Pet Sematary std deviation:       {np.std(pet_scores):.3f}")
print()

print(f"The Shining mean coherence:       {np.mean(shining_scores):.3f}")
print(f"The Shining std deviation:        {np.std(shining_scores):.3f}")
print()

print(f"Overall mean coherence (both):    {np.mean(pet_scores + shining_scores):.3f}")


PARAGRAPH COHERENCE SUMMARY
Pet Sematary paragraphs used:     152
The Shining paragraphs used:      438

Pet Sematary mean coherence:      0.487
Pet Sematary std deviation:       0.064

The Shining mean coherence:       0.504
The Shining std deviation:        0.092

Overall mean coherence (both):    0.499
